<a href="https://colab.research.google.com/github/8786park3-hue/milswai/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!ls

sample_data


In [2]:
from datetime import datetime

# 장비 클래스
class Equipment:
    def __init__(self, name, eq_type, importance):
        self.name = name
        self.eq_type = eq_type
        self.importance = importance
        self.status = "정상"
        self.logs = []

    def update_status(self, status):
        self.status = status
        log = f"{datetime.now()} - 상태 변경: {status}"
        self.logs.append(log)

    def show_info(self):
        return {
            "이름": self.name,
            "종류": self.eq_type,
            "중요도": self.importance,
            "상태": self.status
        }


# 관리 시스템 클래스
class EquipmentManager:
    def __init__(self):
        self.equipments = []

    # 1. 장비 등록
    def add_equipment(self, equipment):
        self.equipments.append(equipment)
        print(f"[등록 완료] {equipment.name}")

    # 2. 장비 목록 조회
    def list_equipments(self):
        for eq in self.equipments:
            print(eq.show_info())

    # 3. 상태 모니터링
    def monitor(self):
        print("\n[모니터링 시작]")
        for eq in self.equipments:
            print(f"{eq.name} 상태: {eq.status}")

    # 4. 장애 처리
    def handle_issue(self, name):
        for eq in self.equipments:
            if eq.name == name:
                eq.update_status("장애 발생")
                print(f"[경고] {name} 장애 발생 → 조치 필요")

    # 5. 유지보수
    def maintenance(self, name):
        for eq in self.equipments:
            if eq.name == name:
                eq.update_status("점검 완료")
                print(f"[유지보수] {name} 점검 완료")

    # 6. 보고서 출력
    def report(self):
        print("\n[운영 보고서]")
        for eq in self.equipments:
            print(f"\n장비: {eq.name}")
            for log in eq.logs:
                print(log)


# 실행 예시 (프로젝트 시뮬레이션)
if __name__ == "__main__":
    manager = EquipmentManager()

    # 장비 등록
    server = Equipment("서버1", "Server", "핵심")
    network = Equipment("스위치1", "Network", "일반")

    manager.add_equipment(server)
    manager.add_equipment(network)

    # 운영
    manager.monitor()

    # 장애 발생
    manager.handle_issue("서버1")

    # 유지보수
    manager.maintenance("서버1")

    # 보고서 출력
    manager.report()

[등록 완료] 서버1
[등록 완료] 스위치1

[모니터링 시작]
서버1 상태: 정상
스위치1 상태: 정상
[경고] 서버1 장애 발생 → 조치 필요
[유지보수] 서버1 점검 완료

[운영 보고서]

장비: 서버1
2026-04-28 07:09:23.623796 - 상태 변경: 장애 발생
2026-04-28 07:09:23.623818 - 상태 변경: 점검 완료

장비: 스위치1


In [4]:
import sqlite3
from datetime import datetime

DB_NAME = "equipment.db"


# DB 초기화
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS equipment (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        type TEXT,
        importance TEXT,
        status TEXT
    )
    """)

    cur.execute("""
    CREATE TABLE IF NOT EXISTS logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        equipment_id INTEGER,
        message TEXT,
        created_at TEXT,
        FOREIGN KEY (equipment_id) REFERENCES equipment(id)
    )
    """)

    conn.commit()
    conn.close()


# 장비 등록
def add_equipment(name, eq_type, importance):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()

    cur.execute("""
    INSERT INTO equipment (name, type, importance, status)
    VALUES (?, ?, ?, ?)
    """, (name, eq_type, importance, "정상"))

    conn.commit()
    conn.close()
    print(f"[등록 완료] {name}")


# 장비 목록 조회
def list_equipment():
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()

    cur.execute("SELECT * FROM equipment")
    rows = cur.fetchall()

    for row in rows:
        print(row)

    conn.close()


# 상태 변경 + 로그 기록
def update_status(equipment_id, status):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()

    cur.execute("""
    UPDATE equipment SET status = ? WHERE id = ?
    """, (status, equipment_id))

    log_message = f"상태 변경: {status}"
    cur.execute("""
    INSERT INTO logs (equipment_id, message, created_at)
    VALUES (?, ?, ?)
    """, (equipment_id, log_message, datetime.now()))

    conn.commit()
    conn.close()
    print(f"[상태 변경] ID {equipment_id} → {status}")


# 장애 처리
def handle_issue(equipment_id):
    update_status(equipment_id, "장애 발생")


# 유지보수
def maintenance(equipment_id):
    update_status(equipment_id, "점검 완료")


# 로그 조회
def show_logs(equipment_id):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()

    cur.execute("""
    SELECT message, created_at
    FROM logs
    WHERE equipment_id = ?
    """, (equipment_id,))

    logs = cur.fetchall()

    print(f"\n[로그 - 장비 ID {equipment_id}]")
    for log in logs:
        print(log)

    conn.close()


# 실행 예시
if __name__ == "__main__":
    init_db()

    # 장비 등록
    add_equipment("서버1", "Server", "핵심")
    add_equipment("스위치1", "Network", "일반")

    # 목록 확인
    list_equipment()

    # 장애 발생
    handle_issue(1)

    # 유지보수
    maintenance(1)

    # 로그 확인
    show_logs(1)

[등록 완료] 서버1
[등록 완료] 스위치1
(1, '서버1', 'Server', '핵심', '정상')
(2, '스위치1', 'Network', '일반', '정상')
[상태 변경] ID 1 → 장애 발생
[상태 변경] ID 1 → 점검 완료

[로그 - 장비 ID 1]
('상태 변경: 장애 발생', '2026-04-28 07:18:02.536108')
('상태 변경: 점검 완료', '2026-04-28 07:18:02.548262')


/tmp/ipykernel_5612/1393328785.py:75: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cur.execute("""


In [ ]:
from flask import Flask, request, redirect, session
import sqlite3
from datetime import datetime

app = Flask(__name__)
app.secret_key = "secret123"
DB = "equipment.db"


# DB 연결
def get_db():
    return sqlite3.connect(DB)


# DB 초기화
def init_db():
    conn = get_db()
    cur = conn.cursor()

    cur.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        password TEXT,
        role TEXT
    )""")

    cur.execute("""CREATE TABLE IF NOT EXISTS equipment (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        type TEXT,
        importance TEXT,
        status TEXT
    )""")

    cur.execute("""CREATE TABLE IF NOT EXISTS logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        equipment_id INTEGER,
        message TEXT,
        created_at TEXT
    )""")

    conn.commit()
    conn.close()


# -------------------------
# 회원가입
# -------------------------
@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]
        role = request.form["role"]

        conn = get_db()
        cur = conn.cursor()
        cur.execute("INSERT INTO users (username, password, role) VALUES (?, ?, ?)",
                    (username, password, role))
        conn.commit()
        conn.close()

        return "회원가입 완료"

    return '''
    <form method="post">
        아이디: <input name="username"><br>
        비밀번호: <input name="password"><br>
        권한:
        <select name="role">
            <option value="user">User</option>
            <option value="admin">Admin</option>
        </select><br>
        <button>가입</button>
    </form>
    '''


# -------------------------
# 로그인
# -------------------------
@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        conn = get_db()
        cur = conn.cursor()
        cur.execute("SELECT * FROM users WHERE username=? AND password=?",
                    (username, password))
        user = cur.fetchone()
        conn.close()

        if user:
            session["user"] = user[1]
            session["role"] = user[3]
            return redirect("/dashboard")
        else:
            return "로그인 실패"

    return '''
    <form method="post">
        아이디: <input name="username"><br>
        비밀번호: <input name="password"><br>
        <button>로그인</button>
    </form>
    '''


# -------------------------
# 대시보드
# -------------------------
@app.route("/dashboard")
def dashboard():
    if "user" not in session:
        return redirect("/login")

    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT * FROM equipment")
    data = cur.fetchall()
    conn.close()

    html = "<h2>장비 목록</h2>"
    for d in data:
        html += f"{d} <br>"

    html += '<br><a href="/add">장비 추가</a>'
    html += '<br><a href="/logout">로그아웃</a>'
    return html


# -------------------------
# 장비 등록 (관리자만)
# -------------------------
@app.route("/add", methods=["GET", "POST"])
def add():
    if session.get("role") != "admin":
        return "관리자만 접근 가능"

    if request.method == "POST":
        name = request.form["name"]
        eq_type = request.form["type"]
        importance = request.form["importance"]

        conn = get_db()
        cur = conn.cursor()
        cur.execute("""INSERT INTO equipment (name, type, importance, status)
                       VALUES (?, ?, ?, ?)""",
                    (name, eq_type, importance, "정상"))
        conn.commit()
        conn.close()

        return redirect("/dashboard")

    return '''
    <form method="post">
        이름: <input name="name"><br>
        종류: <input name="type"><br>
        중요도: <input name="importance"><br>
        <button>추가</button>
    </form>
    '''


# -------------------------
# 상태 변경 (관리자만)
# -------------------------
@app.route("/update/<int:id>")
def update(id):
    if session.get("role") != "admin":
        return "관리자만 가능"

    conn = get_db()
    cur = conn.cursor()

    cur.execute("UPDATE equipment SET status=? WHERE id=?",
                ("점검 완료", id))

    cur.execute("INSERT INTO logs VALUES (NULL, ?, ?, ?)",
                (id, "상태 변경", datetime.now()))

    conn.commit()
    conn.close()

    return redirect("/dashboard")


# -------------------------
# 로그아웃
# -------------------------
@app.route("/logout")
def logout():
    session.clear()
    return redirect("/login")


# 실행
if __name__ == "__main__":
    init_db()
    app.run(debug=True)